# Optional 04 extension — Attribution and feedbacks

**Outside the four-hour core; no required submission.** This notebook contains
Part I's tagged-storage analysis and Part III's feedback hypotheses. The aggregate
OA/OAE response chains and full benchmark figures (former Part II) are now in
[core 04](../04_pump_strength_OA_OAE.ipynb); complete that notebook first.

The extension remains executable in a fresh kernel: supplied prerequisite cells
rebuild the fixed cases for feedback comparisons without repeating the core
exercises or figures. Allow a separate session for this optional material.
The tags are bookkeeping attribution; feedback laws are hypotheses, not observations.

## Roadmap and notation

- **Part I below:** one closed G+S+C trajectory with diagnostic process tags.
- **Core 04:** complete-model carbon/TA forcing and response interpretation.
- **Part III below:** state-dependent pump hypotheses with matched controls.

$G$ means gas exchange, $S$ the soft-tissue pump, and $C$ the carbonate pump.
For tagging only, POC fully remineralizes and PIC fully redissolves at depth.
Feedback experiments restore weathering, dissolution and burial. Temperature,
circulation and mixing retain their benchmark values; $G$ is not a separately
varied solubility-pump sensitivity.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'teaching_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model import initialize_model, postprocess_carbonate_horizons, run_model
from presets import load_boudreau_parameters, make_process_variant, make_pump_variant
from reservoir_inputs import reservoir_inventory_rows
from model_inputs import read_model_tables
from pump_functions import normalized_hill
from scenarios import add_alkalinity_signal, add_carbon_signal
from storage_decomposition import decompose_dic_storage

DATA = ROOT / 'data' / 'Boudreau_2010'
WORKBOOK = DATA / 'model_definition.xlsx'
P = load_boudreau_parameters(WORKBOOK)
input_tables = read_model_tables(WORKBOOK)
STATE = DATA / 'steady_state'
PULSE_FILE = DATA / 'IS92a-scenario.csv'
REFERENCE_SCALE = 0.877
PULSE_START = 1800.0
REFERENCE_CARBON_PMOL = 335.3560189847107
OAE_TARGET_PMOL = 10.0
OAE_SCALE = REFERENCE_SCALE * OAE_TARGET_PMOL / REFERENCE_CARBON_PMOL
from teaching_plots import response_summary

### Reuse the verified model definition from 03

The shared [model workbook](../../../data/Boudreau_2010/model_definition.xlsx) supplies reservoir geometry, water properties, initial states, boundary nodes, transport and gas-exchange arrows, process rates, carbonate choices, and feedback parameters. The tables below show the inputs loaded for this session. Save Excel edits and rerun setup and dependent cells to refresh them. Every case receives an independent copy of the same baseline.

The adapter uses the explicit area and volume columns; no area fractions or depth-based geometry conversion is needed. Box-specific benchmark conditions remain distinct from 01/02.

Keep the supplied benchmark inputs for the reference cases. Part I starts from the workbook concentrations. The supplied fixed cases and Part III load the archived stationary restart, replacing initial ocean concentrations and atmospheric CO₂. Changing a workbook initial concentration therefore does not change those restart runs. Altered geometry, chemistry, connections or process rates requires a new stationary restart and a matched control.

Forcing amounts, pump strengths and feedback switches remain visible in the experiment cells. Feedback equations and conservation checks remain in Python. PIC always links one DIC to two TA equivalents, including when its rate varies with state.

In [ ]:
display(pd.DataFrame(input_tables['OceanReservoirs']).set_index('Box ID'))
display(pd.DataFrame(input_tables['Atmosphere']).set_index('Box ID'))
display(pd.DataFrame(input_tables['BoundaryNodes']).set_index('Box ID'))
display(pd.DataFrame(input_tables['TransportConnections']).sort_values('Order'))
display(pd.DataFrame(input_tables['GasExchangeConnections']).sort_values('Order'))
display(pd.DataFrame(input_tables['ProcessParameters']).set_index('Parameter'))
print('Derived PIC:', P['pic_export'], '; weathering TA:', P['weathering_ta'])

# Part I — Process-tagged carbon-storage decomposition

## 1. Diagnose all processes in one coupled baseline

Turning processes on and off creates different atmospheric equilibria and therefore does not cleanly identify the carbon stored by each process. Instead, solve one physical $G+S+C$ model and carry four diagnostic fields through the same THC and mixing operator $\mathcal{T}$:

$$
C_{\mathrm{DIC}}=C_B+C_G+C_S+C_C.
$$

$C_B$ contains the initial DIC field. The process tags start at zero and receive only their realized source–sink pattern:

$$
\frac{\mathrm{d}C_k}{\mathrm{d}t}
=\mathcal{T}(C_k)+q_k(t),
\qquad k\in\{B,G,S,C\}.
$$

Gas exchange uses the flux generated by the coupled carbonate system and prognostic atmosphere. The biological tags receive the POC and PIC transfers. The tags are passive bookkeeping variables: they diagnose the calculation without feeding back on it.

## 2. Close the attribution boundary

For this diagnostic section only:

- gas exchange, both biological pumps, THC, and high-latitude mixing are all active;
- weathering and burial are disabled;
- all exported POC remineralizes in the deep box;
- all exported PIC redissolves in the deep box with exact 1:2 DIC–TA stoichiometry;
- the finite atmosphere is prognostic.

Thus no process is switched between attribution cases—there is only one case. The `280 ppm` parameter is the initial atmospheric composition, not a fixed boundary condition. Ocean uptake or outgassing changes atmospheric pCO₂, while atmosphere plus ocean carbon remains closed apart from a small numerical integration residual.

## 3. Exercise: predict the tagged contrasts

Before running, predict the signs of the $G$, $S$, and $C$ contributions in the high-latitude and deep boxes relative to the low-latitude surface. Which tags are internally redistributive? Which tag can exchange inventory with the atmosphere? Why does choosing a reference box matter?

> **Your explanation:** replace this placeholder with your answer.

In [ ]:
TAG_LABELS = {
    'gas_exchange': r'$\Delta C_G$ (gas exchange)',
    'soft_tissue': r'$\Delta C_S$ (soft tissue)',
    'carbonate': r'$\Delta C_C$ (carbonate)',
}
TAG_COLORS = {
    'gas_exchange': '#d9d9d9',
    'carbonate': '#9ecae1',
    'soft_tissue': '#969696',
}

assert list(TAG_LABELS) == ['gas_exchange', 'soft_tissue', 'carbonate']


In [ ]:
STORAGE_DURATION = '20 kyr'
storage_params = make_process_variant(
    base=P,
    gas_exchange=True,
    soft_tissue=True,
    carbonate=True,
    weathering=False,
)
storage_model = initialize_model(
    storage_params,
    stop=STORAGE_DURATION,
    max_timestep='10 yr',
)
display(pd.DataFrame(reservoir_inventory_rows(storage_model)).set_index('Box'))
run_model(storage_model)
storage = decompose_dic_storage(storage_model)


## 4. Audit physical conservation and tag closure

The physical active-carbon inventory is

$$
C_{\mathrm{active}}
=C_{\mathrm{atmosphere}}+C_L+C_H+C_D.
$$

The gas reservoir stores a mole fraction, so atmospheric carbon is atmospheric mole inventory multiplied by atmospheric CO₂ mole fraction. For the tags, $S$ and $C$ conserve their ocean sums, while ocean $G$ is balanced by the atmospheric gas-exchange tag:

$$
\sum_i C_{S,i}=0,
\qquad
\sum_i C_{C,i}=0,
\qquad
C_{G,\mathrm{atm}}+\sum_i C_{G,i}=0.
$$

In [ ]:
def active_carbon_inventory(model, index):
    atmospheric = (
        model.CO2_At.c[index]
        * model.CO2_At.reservoir_mass.to('mol').magnitude
    )
    ocean = sum(box.DIC.m[index] for box in (model.L_b, model.H_b, model.D_b))
    return atmospheric + ocean


initial_active = active_carbon_inventory(storage_model, 0)
final_active = active_carbon_inventory(storage_model, -1)
physical_drift = (final_active - initial_active) / initial_active

tag_budget = pd.Series({
    'physical active-C drift': physical_drift,
    'soft-tissue ocean sum (mol)': storage.mass['soft_tissue'][-1].sum(),
    'carbonate ocean sum (mol)': storage.mass['carbonate'][-1].sum(),
    'gas ocean + atmosphere (mol)': (
        storage.mass['gas_exchange'][-1].sum()
        + storage.atmospheric_gas_tag_mol[-1]
    ),
    'maximum final DIC reconstruction error (umol/kg)': (
        np.abs(storage.closure_umol_kg[-1]).max()
    ),
    'initial atmospheric pCO2 (ppm)': storage_model.CO2_At.c[0] * 1e6,
    'final atmospheric pCO2 (ppm)': storage_model.CO2_At.c[-1] * 1e6,
})

assert abs(physical_drift) < 5e-4
assert abs(tag_budget['soft-tissue ocean sum (mol)']) < 1e5
assert abs(tag_budget['carbonate ocean sum (mol)']) < 1e5
assert abs(tag_budget['gas ocean + atmosphere (mol)']) < 1e8
assert tag_budget['maximum final DIC reconstruction error (umol/kg)'] < 0.5
tag_budget


## 5. Choose the reference level and construct the three-box profile

Internal redistribution fixes contrasts, not an absolute zero for each tag. Following the logic of a vertical storage decomposition, define every process contribution relative to the low-latitude source box:

$$
\Delta C_{k,i}=C_{k,i}-C_{k,L},
\qquad k\in\{G,S,C\}.
$$

The low-latitude value is therefore zero by definition. With only three ocean boxes, horizontal stacked bars show the resolved contrasts without inventing a continuous depth profile. The background tracer becomes spatially uniform after 20 kyr, so the tagged sum reconstructs the modeled DIC contrast.

In [ ]:
box_names = {
    'L_b': 'Low-latitude surface',
    'H_b': 'High-latitude surface',
    'D_b': 'Deep ocean',
}
tagged_final = pd.DataFrame(
    {
        tag: storage.concentration_umol_kg[tag][-1]
        for tag in TAG_LABELS
    },
    index=[box_names[name] for name in storage.box_order],
)
storage_contrast = tagged_final.subtract(
    tagged_final.loc['Low-latitude surface'],
    axis='columns',
)
storage_contrast = storage_contrast.rename(columns=TAG_LABELS)

model_final = pd.Series(
    storage.model_dic_umol_kg[-1],
    index=storage_contrast.index,
)
model_contrast = model_final - model_final.loc['Low-latitude surface']

# Exercise 04.1: verify that the three referenced tags reconstruct the
# modeled DIC contrast.
raise NotImplementedError("Exercise: replace this line with your solution")
assert np.abs(contrast_closure).max() < 1e-6

plot_order = [
    TAG_LABELS['gas_exchange'],
    TAG_LABELS['carbonate'],
    TAG_LABELS['soft_tissue'],
]
ax = storage_contrast[plot_order].plot.barh(
    stacked=True,
    figsize=(9, 4.8),
    color=[
        TAG_COLORS['gas_exchange'],
        TAG_COLORS['carbonate'],
        TAG_COLORS['soft_tissue'],
    ],
    edgecolor='black',
    linewidth=0.6,
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(r'DIC storage relative to low-latitude surface ($\mu$mol kg$^{-1}$)')
ax.set_ylabel('')
ax.set_title('Process-tagged three-box carbon-storage decomposition')
ax.legend(title='Tagged process', frameon=False)
plt.tight_layout()
plt.show()

display(storage_contrast)
display(pd.DataFrame({
    'model DIC contrast': model_contrast,
    'sum of tags': tagged_contrast_sum,
    'closure error': contrast_closure,
}))


## 6. Exercise: interpret storage rather than inventory

Explain why $S$ and $C$ can show large positive deep-ocean storage even though each tag conserves its total ocean inventory. Why is $G$ allowed to change total ocean carbon? Why should these bars not be described as three independently switchable equilibria?

> **Your explanation:** replace this placeholder with your answer.

## Supplied prerequisites for Part III

The [core 04 notebook](../04_pump_strength_OA_OAE.ipynb) owns the OA/OAE forcing
questions, response-chain explanations and eight-panel figures. These supplied
cells only rebuild the same fixed-pump control/OA/OAE cases in this fresh kernel
and verify their forcing amounts. They introduce no additional exercise.

The tagged Part I trajectory has a closed boundary and is not the control for
these complete-model experiments. Each feedback case below also needs its own
matching unforced control.

In [ ]:
def build_complete_case(
    forcing=None,
    *,
    soft_feedback=False,
    carbonate_feedback=False,
    max_timestep='1 month',
):
    params = make_pump_variant(
        base=P,
        solubility_strength=1.0,
        soft_tissue_strength=1.0,
        carbonate_strength=1.0,
        soft_tissue_feedback=soft_feedback,
        carbonate_feedback=carbonate_feedback,
    )
    model = initialize_model(params, stop='3800 yr', max_timestep=max_timestep)
    model.read_state(directory=str(STATE))

    if forcing == 'OA':
        add_carbon_signal(
            model,
            filename=PULSE_FILE,
            scale=REFERENCE_SCALE,
            name='reference_carbon_pulse',
        )
    elif forcing == 'OAE':
        add_alkalinity_signal(
            model,
            filename=PULSE_FILE,
            scale=OAE_SCALE,
            name='reference_alkalinity_pulse',
        )
    elif forcing is not None:
        raise ValueError(forcing)
    return model


fixed_cases = {
    'control': build_complete_case(),
    'OA': build_complete_case('OA'),
    'OAE': build_complete_case('OAE'),
}

In [ ]:
def integrated_pulse(signal, time, start=PULSE_START):
    mask = time >= start
    return np.trapezoid(signal.signal_data.m[mask], time[mask])


# Supplied check of the core 04 reference forcing amounts.
oa_moles = integrated_pulse(fixed_cases['OA'].carbon_signal, fixed_cases['OA'].time)
oae_equivalents = integrated_pulse(
    fixed_cases['OAE'].alkalinity_signal,
    fixed_cases['OAE'].time,
)
oa_gtc = oa_moles * 12.0 / 1e15
oae_pmol = oae_equivalents / 1e15

assert np.isclose(oa_gtc, 4025.0, rtol=1e-3)
assert np.isclose(oae_pmol, OAE_TARGET_PMOL, rtol=1e-3)
pd.Series({'OA input (Gt C)': oa_gtc, 'OAE input (Pmol eq)': oae_pmol})

In [ ]:
for label, model in fixed_cases.items():
    print('running complete fixed-pump case', label)
    run_model(model)
    postprocess_carbonate_horizons(model)

for model in fixed_cases.values():
    np.testing.assert_allclose(
        model.D_b.CaCO3_export.c,
        model.D_b.Fdiss.c + model.D_b.Fburial.c,
        rtol=0,
        atol=1e-6,
    )

# Optional extension — Part III: State-dependent biological-pump hypotheses

## 7. Separate prescribed strength from live feedback

All feedback experiments retain unit reference strengths. For either biological pump,

$$
F(t)=F_{\mathrm{ref}}\,g(X(t)),
\qquad g(X_{\mathrm{ref}})=1.
$$

The supplied soft-tissue hypothesis is a bounded Hill response to atmospheric pCO₂,

$$
g_S=
\frac{p\mathrm{CO}_2^n/(K_C^n+p\mathrm{CO}_2^n)}
{p\mathrm{CO}_{2,\mathrm{ref}}^n/(K_C^n+p\mathrm{CO}_{2,\mathrm{ref}}^n)},
$$

and the supplied carbonate hypothesis responds to carbonate excess $A_C=\mathrm{TA}-\mathrm{DIC}$,

$$
g_C=
\frac{A_C^n/(K_A^n+A_C^n)}
{A_{C,\mathrm{ref}}^n/(K_A^n+A_{C,\mathrm{ref}}^n)}.
$$

These are transparent mechanism hypotheses, not calibrated ecosystem models. The `ExternalCode` wiring, normalization, flux reconstruction, and exact PIC DIC–TA stoichiometry are provided rather than assigned.

In [ ]:
x = np.linspace(0.01, 4.0, 300)
response = np.array([normalized_hill(value, 1.0, 1.0, 2.0) for value in x])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, response)
ax.axvline(1, color='0.5', linestyle='--')
ax.axhline(1, color='0.5', linestyle='--')
ax.set(
    xlabel='Driver / reference driver',
    ylabel='Normalized export response',
    title='Provided bounded state-feedback law',
)
ax.grid(alpha=0.15)
plt.show()

## 8. Use a matched 2-by-2 feedback design

| Configuration | Soft-tissue feedback | Carbonate feedback |
| --- | ---: | ---: |
| Fixed | off | off |
| Soft only | on | off |
| Carbonate only | off | on |
| Both | on | on |

Each configuration receives its own unforced, OA, and OAE model. Feedback runs use a one-year maximum step for classroom speed; the fixed-pump, one-month reference reproduction remains the authoritative Figure 4 comparison.

In [ ]:
feedback_switches = {
    'fixed': (False, False),
    'soft only': (True, False),
    'carbonate only': (False, True),
    'both': (True, True),
}

feedback_cases = {
    'fixed': fixed_cases,
}

for label, (soft_feedback, carbonate_feedback) in feedback_switches.items():
    if label == 'fixed':
        continue
    feedback_cases[label] = {}
    for forcing in ('control', 'OA', 'OAE'):
        print('running feedback case', label, forcing)
        model = build_complete_case(
            None if forcing == 'control' else forcing,
            soft_feedback=soft_feedback,
            carbonate_feedback=carbonate_feedback,
            max_timestep='1 yr',
        )
        run_model(model)
        postprocess_carbonate_horizons(model)
        feedback_cases[label][forcing] = model

for label in ('carbonate only', 'both'):
    for model in feedback_cases[label].values():
        np.testing.assert_allclose(
            model.carbonate_ta_connection.fh.m,
            2 * model.CaCO3_export_flux.m,
            rtol=0,
            atol=0,
        )

## 9. Compare feedback effects only through matched anomalies

For feedback configuration $k$,

$$
\Delta Y_k(t)
=Y_{k,\mathrm{forced}}(t)-Y_{k,\mathrm{control}}(t).
$$

Never subtract a fixed-pump control from a feedback-enabled forced case: that would mix the external forcing response with a change in model formulation.

In [ ]:
def plotted_flux(connection, model):
    values = np.asarray(connection.m, dtype=float)
    if values.size == model.time.size:
        return values
    # ESBMTK stores a fixed flux as a two-point series; draw it as constant.
    return np.full(model.time.shape, values[-1], dtype=float)


fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for label, cases in feedback_cases.items():
    for column, forcing in enumerate(('OA', 'OAE')):
        forced = cases[forcing]
        control = cases['control']
        axes[0, column].plot(
            forced.time,
            (forced.CO2_At.c - control.CO2_At.c) * 1e6,
            label=label,
        )
        axes[1, column].plot(
            forced.time,
            forced.L_b.pH.c - control.L_b.pH.c,
            label=label,
        )

for column, forcing in enumerate(('OA', 'OAE')):
    axes[0, column].set_title(forcing)
    axes[1, column].set_xlabel('Model year')
    for axis in axes[:, column]:
        axis.axhline(0, color='black', linewidth=0.7)
        axis.axvline(PULSE_START, color='0.75', linewidth=0.7)
        axis.grid(alpha=0.15)
axes[0, 0].set_ylabel('pCO2 anomaly (ppm)')
axes[1, 0].set_ylabel('Low-latitude pH anomaly')
axes[0, 1].legend(frameon=False, fontsize=8)
fig.suptitle('State-dependent pumps: matched forcing responses')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for label, cases in feedback_cases.items():
    for column, forcing in enumerate(('OA', 'OAE')):
        model = cases[forcing]
        axes[0, column].plot(
            model.time,
            plotted_flux(model.OM_export_flux, model) / 1e12,
            label=label,
        )
        axes[1, column].plot(
            model.time,
            plotted_flux(model.CaCO3_export_flux, model) / 1e12,
            label=label,
        )

for column, forcing in enumerate(('OA', 'OAE')):
    axes[0, column].set_title(forcing)
    axes[1, column].set_xlabel('Model year')
    for axis in axes[:, column]:
        axis.axvline(PULSE_START, color='0.75', linewidth=0.7)
        axis.grid(alpha=0.15)
axes[0, 0].set_ylabel('POC export (Tmol C yr$^{-1}$)')
axes[1, 0].set_ylabel('PIC export (Tmol C yr$^{-1}$)')
axes[0, 1].legend(frameon=False, fontsize=8)
fig.suptitle('Biological export under the supplied feedback hypotheses')
fig.tight_layout()
plt.show()

In [ ]:
feedback_summary = pd.concat(
    {
        label: pd.DataFrame({
            forcing: response_summary(cases[forcing], cases['control'])
            for forcing in ('OA', 'OAE')
        })
        for label, cases in feedback_cases.items()
    },
    names=['feedback configuration', 'metric'],
)
feedback_summary

## 10. Exercise: interpret feedback pathways

For OA and OAE separately, identify which feedback is activated most directly, whether it amplifies or opposes the initial forcing response, and how its effects propagate into atmospheric pCO₂, surface pH, carbonate horizons, dissolution, and burial. Distinguish the imposed response law from the net coupled-model result.

> **Your explanation:** replace this placeholder with your answer.

## 11. Scientific and numerical boundaries

1. The published OA run is a model-reproduction exercise; the 10 Pmol, same-shape OAE input is a mechanistic analogue, not a deployment scenario.
2. The storage stack is a tagged attribution from one trajectory; every tag uses the same realized circulation and gas-exchange state.
3. Storage decomposition uses a closed diagnostic carbonate cycle with complete deep PIC redissolution; OA/OAE restore Boudreau weathering, dissolution, and burial.
4. The biological feedback laws are normalized hypotheses, not observational calibrations.
5. Gas exchange is explicit throughout the forcing experiments. Temperature, THC, and high-latitude mixing sensitivities are deliberately not varied.
6. The one-month fixed-pump calculation is the reference reproduction. Repeat selected feedback cases with a smaller maximum step before drawing quantitative conclusions.
7. Check forcing mass, carbon conservation, PIC 1:2 stoichiometry, and $F_{\mathrm{PIC}}=F_{\mathrm{diss}}+F_{\mathrm{burial}}$ before interpreting plots.